In [1]:
# =========================
# [v2] 22개 라벨링 + 5개 상위묶음 후처리
# =========================
from pathlib import Path
import pandas as pd
import numpy as np
import re, json
import httpx
from openai import OpenAI

# =========================
# Config / Paths  (경로 흐름 유지)
# =========================
BASE_DIR = Path.cwd().parent / "데이터수집" / "data"

OUT_DIR = BASE_DIR / "_outputs_trend"
OUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_CACHE_PATH     = OUT_DIR / "base_l3_cache.csv"         # base -> (label_l3,label_l2) 캐시
AGENDA_OUT_PATH     = OUT_DIR / "agenda_preprocessed.csv"   # 전처리 끝난 의사일정
AGENDA_LABELED_PATH = OUT_DIR / "agenda_labeled_v2.csv"     # v2 라벨 포함 최종

REBUILD_BASE_CACHE = True
BATCH_SIZE = 30

# OpenAI 
from pathlib import Path
from dotenv import load_dotenv
import os

env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("❌ OPENAI_API_KEY를 .env에서 불러오지 못했습니다.")

print("✅ OPENAI_API_KEY 로드 완료:", api_key[:10], "...")

MODEL_LABEL = "gpt-4o-mini"

client = OpenAI(
    api_key=API_KEY,
    http_client=httpx.Client(verify=False, timeout=120),
)

# =========================
# Token Usage Logger
# =========================
USAGE = {"calls":0, "prompt":0, "completion":0, "total":0}

def add_usage(usage):
    if usage is None:
        return
    USAGE["calls"] += 1
    USAGE["prompt"] += getattr(usage, "prompt_tokens", 0)
    USAGE["completion"] += getattr(usage, "completion_tokens", 0)
    USAGE["total"] += getattr(usage, "total_tokens", 0)

def print_usage(prefix="USAGE"):
    print(f"[{prefix}] calls={USAGE['calls']} | prompt={USAGE['prompt']:,} | completion={USAGE['completion']:,} | total={USAGE['total']:,}")

# =========================
# Date / IO Helpers
# =========================
def parse_korean_date(x: str):
    if pd.isna(x):
        return pd.NaT
    s = str(x).strip()
    m = re.search(r"(\d{4})\D+(\d{1,2})\D+(\d{1,2})", s)
    if not m:
        return pd.NaT
    y, mo, d = map(int, m.groups())
    try:
        return pd.Timestamp(y, mo, d)
    except:
        return pd.NaT

def safe_read_excel(fp: Path):
    return pd.read_excel(fp, engine="openpyxl")

def load_all_header_summaries(base_dir: Path):
    files = sorted(base_dir.rglob("*_minutes_header_summary.xlsx"))
    dfs, bad = [], []
    for fp in files:
        try:
            df = safe_read_excel(fp)
            df["source_file"] = fp.name
            df["session_dir"] = fp.parent.name
            dfs.append(df)
        except Exception as e:
            bad.append((str(fp), str(e)))

    if not dfs:
        raise RuntimeError("header_summary 파일을 하나도 못 읽었습니다. 경로/패턴 확인 필요")

    out = pd.concat(dfs, ignore_index=True)

    if "item_text" not in out.columns:
        raise RuntimeError(f"item_text 컬럼이 없습니다. 현재 컬럼: {out.columns.tolist()}")

    out["topic"] = out["item_text"].astype(str)
    out["date_dt"] = out["date"].map(parse_korean_date)
    out = out[out["date_dt"].notna()].copy()

    out["year"] = out["date_dt"].dt.year
    out["quarter"] = out["date_dt"].dt.quarter

    out["session_num"] = pd.to_numeric(out.get("session", np.nan), errors="coerce")
    out["meeting_no"]  = out["meeting_no"].astype(str)

    out["meeting_key"] = (
        out["session_num"].fillna(-1).astype(int).astype(str)
        + "_" + out["meeting_no"]
        + "_" + out["date_dt"].dt.strftime("%Y-%m-%d")
    )

    print(f"loaded: {out.shape} | files={len(files)} | bad={len(bad)}")
    if bad:
        print("bad examples:", bad[:3])
    return out

# =========================
# Topic Preprocess
# =========================
def normalize_topic(t: str) -> str:
    if pd.isna(t):
        return ""
    s = str(t).strip()
    s = re.sub(r"^[o○●▪■□\-–—\s]+", "", s).strip()
    s = re.sub(r"^[가-힣]\.\s*", "", s).strip()
    s = re.sub(r"^\d+\.\s*", "", s).strip()
    s = re.sub(r"[·\.]{2,}\s*\d+\s*$", "", s).strip()
    s = re.sub(r"\s+", " ", s).strip()
    return s

def to_base_topic(topic: str) -> str:
    s = normalize_topic(topic)
    s = re.sub(r"\([^)]*의안번호[^)]*\)", "", s).strip()
    s = re.sub(r"\([^)]*(대표발의|정부 제출|의원 대표발의|서면동의|추가)\s*[^)]*\)", "", s).strip()
    s = re.sub(r"\s+", " ", s).strip()
    return s

def rough_topic_type(topic: str) -> str:
    t = normalize_topic(topic)
    if re.search(r"(의안번호|법률안|개정법률안|특별법안|대안|결의안|동의안|승인안|건의안|규칙안|청원)", t):
        return "법안/의안"
    if re.search(r"(국정감사|국정조사|인사청문|업무보고|현안\s*질의|질의|답변|보고)", t):
        return "회의행위"
    if re.search(r"(부|청|처|위원회|원)$", t):
        return "기관"
    return "기타"

# =========================
# L3(22) + L2(5) mapping (하드코딩) - UPDATED
# =========================
L2_GROUPS = [
    "재난·안전",
    "정부혁신·행정지원",
    "인공지능·디지털 정부",
    "지방행정",
    "지방재정·기타",
]

L3_LABELS = [
    # 재난·안전 (7)
    "과학수사", "국민안전행정지원", "복구지원", "비상대비", "안전및재난", "재난안전교육·연구", "재난예방",
    # 정부혁신·행정지원 (6)
    "국가적중장기과제추진·지원", "정부의전", "정부혁신", "조직관리", "청사관리", "안전행정행정지원",
    # 인공지능·디지털 정부 (3)
    "전자정부", "정보시스템통합관리", "국가기록물",
    # 지방행정 (2)
    "지방행정", "자치인력개발",
    # 지방재정·기타 (4)
    "지방재정", "지방세", "지역균형발전", "이북5도",
]
L3_ENUM_STR = ", ".join(L3_LABELS)

L3_TO_L2 = {
    # 재난·안전
    "과학수사": "재난·안전",
    "국민안전행정지원": "재난·안전",
    "복구지원": "재난·안전",
    "비상대비": "재난·안전",
    "안전및재난": "재난·안전",
    "재난안전교육·연구": "재난·안전",
    "재난예방": "재난·안전",

    # 정부혁신·행정지원
    "국가적중장기과제추진·지원": "정부혁신·행정지원",
    "정부의전": "정부혁신·행정지원",
    "정부혁신": "정부혁신·행정지원",
    "조직관리": "정부혁신·행정지원",
    "청사관리": "정부혁신·행정지원",
    "안전행정행정지원": "정부혁신·행정지원",

    # 인공지능·디지털 정부
    "전자정부": "인공지능·디지털 정부",
    "정보시스템통합관리": "인공지능·디지털 정부",
    "국가기록물": "인공지능·디지털 정부",

    # 지방행정
    "지방행정": "지방행정",
    "자치인력개발": "지방행정",

    # 지방재정·기타
    "지방재정": "지방재정·기타",
    "지방세": "지방재정·기타",
    "이북5도": "지방재정·기타",

    # ✅ 변경: 지역균형발전 -> 지방행정(4번)
    "지역균형발전": "지방행정",
}


def coerce_l3(label_raw: str, base: str) -> str:
    """모델이 '재난안전' 같은 변형 라벨을 내면 22개 중 하나로 교정"""
    s = normalize_label(label_raw)

    # 가장 문제되는 케이스: '재난안전' 류 -> 포괄 라벨로 고정
    if re.search(r"(재난\s*안전|재난안전)", s):
        return "안전및재난"

    return s

def call_gpt_label_one_to_l3(base: str, model=MODEL_LABEL) -> dict:
    sys = (
        "아래 base를 22개 라벨(L3) 중 하나로만 분류하세요.\n"
        f"라벨은 반드시 아래 목록에서 '완전 동일 문자열' 1개를 복사해 출력해야 합니다:\n{L3_ENUM_STR}\n"
        "요약/변형/줄임말 금지.\n"
        "JSON만 출력:\n"
        "{\"label_l3\":...,\"confidence\":...,\"rationale\":...,\"evidence\":...}"
    )
    resp = client.chat.completions.create(
        model=model,
        temperature=0,
        response_format={"type":"json_object"},
        messages=[
            {"role":"system","content": sys},
            {"role":"user","content": json.dumps({"base": base}, ensure_ascii=False)},
        ],
    )
    add_usage(resp.usage)
    return json.loads(resp.choices[0].message.content)


def assert_valid_l3(df: pd.DataFrame, col="label_l3"):
    bad = df[~df[col].isin(L3_LABELS)]
    if len(bad) > 0:
        ex = bad[["base", col]].head(10)
        raise RuntimeError(f"L3가 22개 enum 밖으로 나왔습니다. 예시:\n{ex.to_string(index=False)}")

# =========================
# Cache Load/Save (v2)
# =========================
def load_base_cache(path=BASE_CACHE_PATH):
    p = Path(path)
    if p.exists():
        df = pd.read_csv(p, encoding="utf-8-sig")
        for c in ["base","label_l3","label_l2","confidence","rationale","evidence","source"]:
            if c not in df.columns:
                df[c] = np.nan
        return df[["base","label_l3","label_l2","confidence","rationale","evidence","source"]].copy()

    return pd.DataFrame({
        "base": pd.Series(dtype="string"),
        "label_l3": pd.Series(dtype="string"),
        "label_l2": pd.Series(dtype="string"),
        "confidence": pd.Series(dtype="float"),
        "rationale": pd.Series(dtype="string"),
        "evidence": pd.Series(dtype="string"),
        "source": pd.Series(dtype="string"),
    })

def save_base_cache(df: pd.DataFrame, path=BASE_CACHE_PATH):
    df = df.drop_duplicates(subset=["base"], keep="last")
    df.to_csv(path, index=False, encoding="utf-8-sig")




In [2]:
# =========================
# GPT Labeler (base -> L3 22개 강제)
# =========================
def normalize_label(s):
    if not isinstance(s, str):
        return ""
    return re.sub(r"\s+", " ", s.strip())

def call_gpt_label_bases_to_l3(bases, model=MODEL_LABEL):
    items = [{"base": b} for b in bases]

    sys = (
        "당신은 대한민국 국회 '의사일정' 항목(base)을 아래 22개 라벨(L3) 중 하나로 분류하는 라벨러입니다.\n"
        "규칙:\n"
        f"- L3는 반드시 아래 22개 중 하나만 정확히 선택(다른 값/빈값 금지):\n{L3_ENUM_STR}\n"
        "- label_l3는 반드시 위 목록에서 '복사-붙여넣기'한 값이어야 하며, 요약/변형/줄임말은 금지.\n"
        "- 애매해도 22개 중 가장 가까운 것을 반드시 하나 고르세요.\n"
        "- confidence는 0~1.\n"
        "- rationale은 1문장.\n"
        "- evidence는 base에서 근거가 된 핵심 구절(짧게).\n"
        "반드시 JSON만 출력:\n"
        "{\"labels\":[{\"base\":...,\"label_l3\":...,\"confidence\":...,\"rationale\":...,\"evidence\":...}]}"
    )

    resp = client.chat.completions.create(
        model=model,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role":"system","content": sys},
            {"role":"user","content": json.dumps({"items": items}, ensure_ascii=False)},
        ],
    )
    add_usage(resp.usage)
    data = json.loads(resp.choices[0].message.content)
    return data["labels"]


In [3]:
# =========================
# Build Cache (v2)  base -> label_l3 -> label_l2(후처리)
# 핵심 수정:
#  - 캐시 키(base)는 모델 응답 r["base"]를 믿지 말고, 요청한 batch의 base로 강제 저장
#  - 목록 밖 label_l3는 coerce -> (필요시) 단건 재질의 1회 -> fallback
# =========================
def build_base_l3_cache(agenda_df: pd.DataFrame, batch_size=BATCH_SIZE, model=MODEL_LABEL):
    if REBUILD_BASE_CACHE and BASE_CACHE_PATH.exists():
        BASE_CACHE_PATH.unlink()

    bases = sorted(set(agenda_df["base"].astype(str).tolist()))
    base_cache = load_base_cache(BASE_CACHE_PATH)

    cached = set(base_cache["base"].dropna().astype(str).tolist())
    need = [b for b in bases if b not in cached]

    print(f"유니크 base: {len(bases):,} | 캐시됨: {len(cached):,} | 신규 대상(base): {len(need):,}")

    for i in range(0, len(need), batch_size):
        batch = need[i:i+batch_size]

        # 배치 호출
        labels = call_gpt_label_bases_to_l3(batch, model=model)
        if not isinstance(labels, list):
            raise RuntimeError(f"labels 타입이 list가 아닙니다: {type(labels)}")

        rows = []

        # ✅ 가장 중요: "요청한 batch 순서" 기준으로 base를 강제
        # 보통은 labels 길이가 batch와 같고 순서도 맞습니다(대부분 이걸로 해결됨).
        if len(labels) == len(batch):
            pairs = list(zip(batch, labels))
        else:
            # 길이가 안 맞을 때만 방어적으로 매칭(모자라면 단건 재질의로 채움)
            ret_map = {}
            for r in labels:
                k = str(r.get("base", "")).strip()
                if k:
                    ret_map[k] = r

            pairs = []
            for b in batch:
                r = ret_map.get(b)
                if r is None:
                    # 누락된 건은 단건으로 채움
                    one = call_gpt_label_one_to_l3(b, model=model)
                    r = {"base": b, **one}
                pairs.append((b, r))

        for base_in, r in pairs:
            base_in = str(base_in)

            # 1) 기본(배치 응답) 값들
            l3_raw = r.get("label_l3", "") or ""
            l3 = coerce_l3(l3_raw, base_in)   # ex) '재난안전' -> '안전및재난'
            conf = float(r.get("confidence", np.nan)) if r.get("confidence") is not None else np.nan
            rat  = r.get("rationale", "") or ""
            evd  = r.get("evidence", "") or ""
            src  = f"gpt:{model}"

            # 2) 교정 후에도 목록 밖이면, 그 건만 1회 재질의
            if l3 not in L3_TO_L2:
                retry = call_gpt_label_one_to_l3(base_in, model=model)
                l3 = coerce_l3(retry.get("label_l3", ""), base_in)

                if l3 in L3_TO_L2:
                    conf = float(retry.get("confidence", np.nan)) if retry.get("confidence") is not None else np.nan
                    rat  = retry.get("rationale", "") or ""
                    evd  = retry.get("evidence", "") or ""
                    src  = f"gpt:{model}:retry"
                else:
                    # 3) 그래도 목록 밖이면 fallback (절대 중단 금지)
                    l3 = "안전및재난"
                    conf = 0.0
                    rat  = "fallback_due_to_invalid_label"
                    evd  = ""
                    src  = f"gpt:{model}:fallback"

            # ✅ label_l2는 후처리(고정 매핑)
            rows.append({
                "base": base_in,                 # ✅ 여기! base는 무조건 입력값으로 저장
                "label_l3": l3,
                "label_l2": L3_TO_L2[l3],
                "confidence": conf,
                "rationale": rat,
                "evidence": evd,
                "source": src
            })

        base_cache = pd.concat([base_cache, pd.DataFrame(rows)], ignore_index=True)

        # 배치마다 저장(중간 종료돼도 캐시 남음)
        save_base_cache(base_cache, BASE_CACHE_PATH)
        print_usage(prefix=f"GPT BATCH {i//batch_size+1}")

    base_cache = load_base_cache(BASE_CACHE_PATH).drop_duplicates(subset=["base"], keep="last")

    # 최종 검증: label_l3가 22개 밖이면 여기서 잡힘
    assert_valid_l3(base_cache, col="label_l3")

    print_usage(prefix="BASE TOTAL")
    return base_cache


# =========================
# Run (Load -> Agenda -> Cache -> Merge -> Save v2)
# =========================
headers_all = load_all_header_summaries(BASE_DIR)

agenda_df = headers_all[headers_all["section"].astype(str).eq("의사일정")].copy()
agenda_df["topic_norm"] = agenda_df["topic"].map(normalize_topic)
agenda_df["base"] = agenda_df["topic"].map(to_base_topic)
agenda_df["topic_type"] = agenda_df["topic"].map(rough_topic_type)

print("agenda_df:", agenda_df.shape, "| meetings:", agenda_df["meeting_key"].nunique())
print("unique base:", agenda_df["base"].nunique())

agenda_df.to_csv(AGENDA_OUT_PATH, index=False, encoding="utf-8-sig")
print("saved:", AGENDA_OUT_PATH)

base_cache_df = build_base_l3_cache(agenda_df, batch_size=BATCH_SIZE, model=MODEL_LABEL)
print("saved:", BASE_CACHE_PATH)

agenda_labeled = agenda_df.merge(
    base_cache_df[["base","label_l2","label_l3","confidence","rationale","evidence","source"]],
    on="base", how="left"
)

# 누락 있으면 에러
missing = agenda_labeled[agenda_labeled["label_l3"].isna()][["base","topic_norm"]].drop_duplicates().head(20)
if len(missing) > 0:
    raise RuntimeError("라벨 누락이 있습니다. 예시:\n" + missing.to_string(index=False))

# 최종 검증
assert_valid_l3(agenda_labeled.rename(columns={"label_l3":"label_l3"}), col="label_l3")

agenda_labeled.to_csv(AGENDA_LABELED_PATH, index=False, encoding="utf-8-sig")
print("saved:", AGENDA_LABELED_PATH)

print("\n[L2(5)]")
print(agenda_labeled["label_l2"].value_counts())

print("\n[L3(22)]")
print(agenda_labeled["label_l3"].value_counts())

print("\n[topic_type]")
print(agenda_labeled["topic_type"].value_counts())

agenda_labeled.head(5)

loaded: (19887, 15) | files=258 | bad=0
agenda_df: (9828, 18) | meetings: 243
unique base: 1126
saved: C:\Users\user\Desktop\코드\데이터수집\data\_outputs_trend\agenda_preprocessed.csv
유니크 base: 1,126 | 캐시됨: 0 | 신규 대상(base): 1,126
[GPT BATCH 1] calls=1 | prompt=1,029 | completion=2,361 | total=3,390
[GPT BATCH 2] calls=2 | prompt=1,918 | completion=4,417 | total=6,335
[GPT BATCH 3] calls=3 | prompt=2,819 | completion=6,440 | total=9,259
[GPT BATCH 4] calls=4 | prompt=3,721 | completion=8,557 | total=12,278
[GPT BATCH 5] calls=5 | prompt=4,664 | completion=10,773 | total=15,437
[GPT BATCH 6] calls=6 | prompt=5,784 | completion=13,031 | total=18,815
[GPT BATCH 7] calls=7 | prompt=6,763 | completion=15,688 | total=22,451
[GPT BATCH 8] calls=9 | prompt=8,085 | completion=18,118 | total=26,203
[GPT BATCH 9] calls=10 | prompt=9,071 | completion=20,448 | total=29,519
[GPT BATCH 10] calls=11 | prompt=10,176 | completion=23,227 | total=33,403
[GPT BATCH 11] calls=12 | prompt=11,181 | completion=25,506

,session,session_type,meeting_no,date,section,item_order,item_text,source_file,session_dir,topic,...,meeting_key,topic_norm,base,topic_type,label_l2,label_l3,confidence,rationale,evidence,source
0,353.0,임시회,제1호,2017년 8월 21일(월),의사일정,1.0,2016회계연도 결산,제1호_minutes_header_summary.xlsx,제353회,2016회계연도 결산,...,353_제1호_2017-08-21,2016회계연도 결산,2016회계연도 결산,기타,지방재정·기타,지방재정,0.7,회계연도 결산은 지방 재정 관리와 관련이 있다.,2016회계연도 결산,gpt:gpt-4o-mini
1,353.0,임시회,제1호,2017년 8월 21일(월),의사일정,2.0,가. 행정안전부 소관,제1호_minutes_header_summary.xlsx,제353회,가. 행정안전부 소관,...,353_제1호_2017-08-21,행정안전부 소관,행정안전부 소관,기타,정부혁신·행정지원,정부혁신,0.9,행정안전부의 소관은 정부의 혁신과 관련이 있다.,행정안전부 소관,gpt:gpt-4o-mini
2,353.0,임시회,제1호,2017년 8월 21일(월),의사일정,3.0,나. 인사혁신처 소관,제1호_minutes_header_summary.xlsx,제353회,나. 인사혁신처 소관,...,353_제1호_2017-08-21,인사혁신처 소관,인사혁신처 소관,기타,정부혁신·행정지원,정부혁신,0.6,인사혁신처는 정부의 혁신과 관련된 기관이다.,인사혁신처 소관,gpt:gpt-4o-mini
3,353.0,임시회,제1호,2017년 8월 21일(월),의사일정,4.0,다. 경찰청 소관,제1호_minutes_header_summary.xlsx,제353회,다. 경찰청 소관,...,353_제1호_2017-08-21,경찰청 소관,경찰청 소관,기타,재난·안전,국민안전행정지원,0.9,경찰청 소관 사항은 국민의 안전과 관련이 있다.,경찰청 소관,gpt:gpt-4o-mini
4,353.0,임시회,제1호,2017년 8월 21일(월),의사일정,5.0,라. 소방청 소관,제1호_minutes_header_summary.xlsx,제353회,라. 소방청 소관,...,353_제1호_2017-08-21,소방청 소관,소방청 소관,기타,재난·안전,국민안전행정지원,0.9,소방청은 국민의 안전을 담당하는 기관으로 관련된 내용이다.,소방청 소관,gpt:gpt-4o-mini


In [4]:
import pandas as pd

# agenda_labeled 가 이미 만들어진 상태라고 가정

# 1) 소수점 붙은 정수형 컬럼들을 "정수"로 강제 변환
int_cols = ["session", "item_order", "year", "quarter", "session_num"]  # 있으면 변환, 없으면 자동 무시

for c in int_cols:
    if c in agenda_labeled.columns:
        agenda_labeled[c] = (
            pd.to_numeric(agenda_labeled[c], errors="coerce")
              .round(0)
              .astype("Int64")   # 결측 허용 정수 타입
        )

# 2) CSV 저장
agenda_labeled.to_csv(AGENDA_LABELED_PATH, index=False, encoding="utf-8")
print("saved:", AGENDA_LABELED_PATH)

saved: C:\Users\user\Desktop\코드\데이터수집\data\_outputs_trend\agenda_labeled_v2.csv


In [5]:
bad = agenda_labeled["item_text"].astype(str).str.contains("�", na=False).sum()
print("저장 직전 깨진 행 수:", bad)

저장 직전 깨진 행 수: 0


In [6]:
import pandas as pd
import numpy as np

# 기존 라벨 결과 로드
agenda_labeled = pd.read_csv(AGENDA_LABELED_PATH, encoding="utf-8-sig")

# ✅ 새 매핑으로 L2 재계산 (L3는 그대로)
agenda_labeled["label_l2"] = agenda_labeled["label_l3"].map(L3_TO_L2)

# 매핑 누락 검증
miss = agenda_labeled[agenda_labeled["label_l2"].isna()][["label_l3"]].drop_duplicates()
if len(miss) > 0:
    raise RuntimeError("L3_TO_L2에 없는 label_l3가 있습니다:\n" + miss.to_string(index=False))

# 저장 (파일명 원하는 대로)
AGENDA_LABELED_V3_PATH = OUT_DIR / "agenda_labeled_v3.csv"
agenda_labeled.to_csv(AGENDA_LABELED_V3_PATH, index=False, encoding="utf-8-sig")
print("saved:", AGENDA_LABELED_V3_PATH)

# 분포 확인
print("\n[L2]")
print(agenda_labeled["label_l2"].value_counts())

print("\n[L3]")
print(agenda_labeled["label_l3"].value_counts())

saved: C:\Users\user\Desktop\코드\데이터수집\data\_outputs_trend\agenda_labeled_v3.csv

[L2]
label_l2
재난·안전          2887
지방재정·기타        2689
지방행정           2061
정부혁신·행정지원      2000
인공지능·디지털 정부     191
Name: count, dtype: int64

[L3]
label_l3
안전및재난            2222
지방세              2085
지방행정             1750
정부혁신             1535
지방재정              602
국민안전행정지원          578
지역균형발전            281
정부의전              225
조직관리              130
국가적중장기과제추진·지원      97
전자정부               89
정보시스템통합관리          60
국가기록물              42
비상대비               40
자치인력개발             30
복구지원               24
재난예방               21
안전행정행정지원           12
과학수사                2
이북5도                2
청사관리                1
Name: count, dtype: int64


In [7]:
import pandas as pd

# 0) 매핑이 진짜 새 버전인지 즉시 검증 (여기서 실패하면 매핑 셀부터 다시 실행)
print("안전및재난 ->", L3_TO_L2["안전및재난"])         # 기대: 재난·안전
print("정부혁신 ->", L3_TO_L2["정부혁신"])             # 기대: 정부혁신·행정지원
print("국가기록물 ->", L3_TO_L2["국가기록물"])         # 기대: 인공지능·디지털 정부
print("지역균형발전 ->", L3_TO_L2["지역균형발전"])     # 기대: 지방행정

# 1) 기존 라벨 파일 로드 (v2든 v3든 상관 없음)
df = pd.read_csv(OUT_DIR / "agenda_labeled_v2.csv", encoding="utf-8-sig")  # <- 안전하게 v2에서 다시 만듦

# 2) L2 재계산
df["label_l2"] = df["label_l3"].map(L3_TO_L2)

# 3) 누락 검증 (하나라도 NaN이면 매핑 딕셔너리 문제)
miss = df[df["label_l2"].isna()]["label_l3"].drop_duplicates()
if len(miss) > 0:
    raise RuntimeError("L3_TO_L2 매핑 누락 L3:\n" + miss.to_string(index=False))

# 4) 저장
out_path = OUT_DIR / "agenda_labeled_v3.csv"
df.to_csv(out_path, index=False, encoding="utf-8-sig")
print("saved:", out_path)

# 5) 분포 확인 (여기서 새 L2 이름으로 나와야 정상)
print("\n[L2]")
print(df["label_l2"].value_counts())

안전및재난 -> 재난·안전
정부혁신 -> 정부혁신·행정지원
국가기록물 -> 인공지능·디지털 정부
지역균형발전 -> 지방행정
saved: C:\Users\user\Desktop\코드\데이터수집\data\_outputs_trend\agenda_labeled_v3.csv

[L2]
label_l2
재난·안전          2887
지방재정·기타        2689
지방행정           2061
정부혁신·행정지원      2000
인공지능·디지털 정부     191
Name: count, dtype: int64


In [8]:
import pandas as pd

df = pd.read_csv(OUT_DIR / "agenda_preprocessed.csv", encoding="utf-8-sig")
df["date_dt"] = pd.to_datetime(df["date_dt"], errors="coerce")

by_dir = df.groupby("session_dir").agg(
    rows=("base","size"),
    min_date=("date_dt","min"),
    max_date=("date_dt","max"),
    sessions=("session_num","nunique"),
).sort_values("min_date")

print(by_dir.tail(30))

             rows   min_date   max_date  sessions
session_dir                                      
제391회         575 2021-09-08 2021-12-07         1
제429회         439 2021-12-07 2025-11-27         2
제392회          65 2022-01-05 2022-01-06         1
제393회          93 2022-01-27 2022-02-14         1
제394회          69 2022-04-05 2022-04-05         1
제395회           7 2022-04-22 2022-04-28         1
제397회          40 2022-05-03 2022-05-16         1
제398회           6 2022-07-28 2022-08-08         1
제399회          18 2022-08-18 2022-08-29         1
제400회         615 2022-09-01 2022-12-01         1
제403회         342 2023-02-15 2023-02-24         1
제404회         111 2023-03-13 2023-03-22         1
제405회         163 2023-04-25 2023-04-25         1
제406회         140 2023-05-16 2023-05-24         1
제407회          97 2023-06-22 2023-06-22         1
제408회           2 2023-07-13 2023-07-13         1
제409회           3 2023-08-23 2023-08-31         1
제410회         659 2023-09-20 2023-12-08         1


In [9]:
by_dir

,rows,min_date,max_date,sessions
session_dir,,,,
제353회,26,2017-08-21,2017-08-23,1
제354회,426,2017-09-18,2017-11-30,1
제355회,25,2017-12-20,2018-01-10,1
제356회,202,2018-01-31,2018-02-22,1
제358회,28,2018-03-27,2018-03-27,1
...,...,...,...,...
제420회,89,2024-12-23,2025-01-13,1
제421회,1,2025-01-20,2025-01-20,1
제422회,56,2025-02-19,2025-02-27,1
